# 3 · XGBoost — Predicting Flight Delays

XGBoost is a **gradient-boosted tree** algorithm. Unlike Random Forest, which averages independent trees, XGBoost builds trees **sequentially** — each new tree tries to correct the residual errors of the ensemble so far. It also adds explicit regularization and very efficient tree-construction, which is why it dominates structured-data competitions.

Compared to Random Forest you should expect:
- Slightly better ranking metrics (ROC / PR AUC)
- More sensitivity to hyperparameters
- Faster training on large data thanks to histogram-based splits

## Setup

**Required packages**
```
pip install pandas numpy matplotlib seaborn scikit-learn xgboost
```
The dataset is `bwi_model_ready.csv` — already cleaned by `preprocess.py`.
It covers 14 days at Baltimore/Washington International (BWI), joined with hourly weather.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
plt.rcParams["figure.figsize"] = (9, 5)

CATEGORICAL = ["carrier", "origin_airport", "destination_airport", "direction"]

df = pd.read_csv("bwi_model_ready.csv")
for col in CATEGORICAL:
    df[col] = df[col].astype("category")

print("shape:", df.shape)
df.head()


### Let XGBoost handle categoricals natively
XGBoost ≥1.7 supports pandas `category` dtype directly — you enable it via `enable_categorical=True`. This keeps our feature table compact (no one-hot blowup) and often yields slightly better splits on high-cardinality columns like destination.

In [ ]:
# Target: binary "is the flight delayed >= 15 min?"
# We drop rows where the label is missing (cancelled flights with no arrival time).
data = df.dropna(subset=["is_delayed"]).copy()
data["is_delayed"] = data["is_delayed"].astype(int)

TARGET = "is_delayed"
DROP_FROM_X = ["is_cancelled", "arrival_delay_minutes", "is_delayed"]
feature_cols = [c for c in data.columns if c not in DROP_FROM_X]

# Time-based split: BTS rows come pre-sorted by date.
# Using a chronological holdout is the right call for this kind of data —
# a random split would leak "future" information into the training set.
split = int(len(data) * 0.8)
train, test = data.iloc[:split], data.iloc[split:]
X_train, y_train = train[feature_cols], train[TARGET]
X_test, y_test = test[feature_cols], test[TARGET]

print(f"train rows: {len(X_train):,}   positive rate: {y_train.mean():.1%}")
print(f"test  rows: {len(X_test):,}   positive rate: {y_test.mean():.1%}")


### Fit the model
Starting hyperparameters (sensible defaults, not tuned):
- `n_estimators=400`, `learning_rate=0.05` — smaller steps × more trees
- `max_depth=6` — moderately deep trees
- `scale_pos_weight` handles class imbalance (ratio of negatives to positives)
- `early_stopping_rounds` stops when the validation metric stops improving

In [ ]:
from xgboost import XGBClassifier

# Split train further into train/val so we can early-stop honestly.
val_split = int(len(X_train) * 0.85)
X_tr, X_val = X_train.iloc[:val_split], X_train.iloc[val_split:]
y_tr, y_val = y_train.iloc[:val_split], y_train.iloc[val_split:]

pos_weight = (y_tr == 0).sum() / max((y_tr == 1).sum(), 1)

model = XGBClassifier(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=5,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    scale_pos_weight=pos_weight,
    enable_categorical=True,
    tree_method='hist',
    eval_metric='aucpr',
    early_stopping_rounds=25,
    random_state=42,
    n_jobs=-1,
)

model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
print(f"best iteration: {model.best_iteration}")
print(f"best PR-AUC on val: {model.best_score:.4f}")

### Metrics

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    roc_curve, precision_recall_curve,
)

proba = model.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

metrics = {
    "accuracy":  accuracy_score(y_test, pred),
    "precision": precision_score(y_test, pred, zero_division=0),
    "recall":    recall_score(y_test, pred, zero_division=0),
    "f1":        f1_score(y_test, pred, zero_division=0),
    "roc_auc":   roc_auc_score(y_test, proba),
    "pr_auc":    average_precision_score(y_test, proba),
}
pd.Series(metrics).round(4).to_frame("value")


In [ ]:
cm = confusion_matrix(y_test, pred)
fig, ax = plt.subplots(figsize=(4.5, 4))
ax.imshow(cm, cmap="Blues")
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                color="white" if cm[i, j] > cm.max()/2 else "black")
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(["on-time", "delayed"])
ax.set_yticklabels(["on-time", "delayed"])
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title("Confusion matrix")
plt.tight_layout(); plt.show()


In [ ]:
fpr, tpr, _ = roc_curve(y_test, proba)
prec, rec, _ = precision_recall_curve(y_test, proba)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(fpr, tpr, label=f"AUC = {metrics['roc_auc']:.3f}")
axes[0].plot([0, 1], [0, 1], "--", color="gray")
axes[0].set(title="ROC curve", xlabel="False positive rate", ylabel="True positive rate")
axes[0].legend()

axes[1].plot(rec, prec, label=f"AP = {metrics['pr_auc']:.3f}")
axes[1].axhline(y_test.mean(), linestyle="--", color="gray",
                label=f"baseline = {y_test.mean():.2f}")
axes[1].set(title="Precision–Recall curve", xlabel="Recall", ylabel="Precision")
axes[1].legend()
plt.tight_layout(); plt.show()


### Feature importance
XGBoost exposes multiple flavors of importance:
- **gain**: total reduction in loss from splits on this feature (usually the most useful)
- **weight**: number of times the feature is used to split
- **cover**: average number of samples affected

Prefer `gain` unless you have a specific reason not to.

In [ ]:
from xgboost import plot_importance

fig, ax = plt.subplots(figsize=(7, 8))
plot_importance(model, max_num_features=20, importance_type='gain', ax=ax)
ax.set_title('Top 20 feature importances (gain)')
plt.tight_layout(); plt.show()

### Training-curve diagnostics
Because we used early stopping, the model picked its own number of trees. The evaluation history tells you whether you're under-fitting (both curves still dropping) or over-fitting (val curve turning up while train keeps improving).

In [ ]:
hist = model.evals_result()
plt.plot(hist['validation_0']['aucpr'], label='val PR-AUC')
plt.axvline(model.best_iteration, color='red', ls='--',
            label=f'best iter = {model.best_iteration}')
plt.xlabel('boosting round'); plt.ylabel('PR-AUC')
plt.title('XGBoost validation curve')
plt.legend(); plt.tight_layout(); plt.show()

### Bonus: regressing the delay magnitude

In [ ]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, r2_score

reg_data = df.dropna(subset=['arrival_delay_minutes']).copy()
sp = int(len(reg_data) * 0.8)
X_reg_tr, X_reg_te = reg_data.iloc[:sp][feature_cols], reg_data.iloc[sp:][feature_cols]
y_reg_tr, y_reg_te = reg_data.iloc[:sp]['arrival_delay_minutes'], reg_data.iloc[sp:]['arrival_delay_minutes']

reg = XGBRegressor(
    n_estimators=400, learning_rate=0.05, max_depth=6,
    subsample=0.9, colsample_bytree=0.9,
    enable_categorical=True, tree_method='hist',
    random_state=42, n_jobs=-1,
)
reg.fit(X_reg_tr, y_reg_tr)
pred = reg.predict(X_reg_te)
print(f"MAE: {mean_absolute_error(y_reg_te, pred):.2f} min")
print(f"R²:  {r2_score(y_reg_te, pred):.3f}")

### Writeup prompts
- How does XGBoost's PR-AUC compare with the Random Forest on the same split?
- Did early stopping trigger? Did the val curve plateau or turn up?
- Which feature importance here matches the Random Forest result? Which diverge, and why might that be?